Here, I try to read in the data.

In [19]:
import pandas as pd
from pathlib import Path

RAW = Path("data/original_data/oecd_regional_2016")
datasets = sorted(RAW.glob("*.csv"))
print(datasets[5].name)
print(sorted(p.name for p in RAW.glob("*.csv")))



WELLBEING_FILE = "RWB-2016-1-EN-20161128T112241.csv"   # replace with the real name

wb_raw = pd.read_csv(RAW / WELLBEING_FILE)

print(wb_raw.shape)
print(wb_raw.columns.tolist())
wb_raw.head()

RWB-2016-1-EN-20161128T112241.csv
['REGION_DEMOGR-2016-1-EN-20161128T111326.csv', 'REGION_ECONOM-2016-1-EN-20161128T111738.csv', 'REGION_INNOVATION-2016-1-EN-20161128T112220.csv', 'REGION_LABOUR-2016-1-EN-20161128T112125.csv', 'REGION_SOCIAL-2016-1-EN-20161128T112210.csv', 'RWB-2016-1-EN-20161128T112241.csv']
(16963, 17)
['REG_ID', 'Regions', 'IND', 'Indicator', 'MEAS', 'Measure', 'TIME', 'Time', 'Unit Code', 'Unit', 'PowerCode Code', 'PowerCode', 'Reference Period Code', 'Reference Period', 'Value', 'Flag Codes', 'Flags']


,REG_ID,Regions,IND,Indicator,MEAS,Measure,TIME,Time,Unit Code,Unit,PowerCode Code,PowerCode,Reference Period Code,Reference Period,Value,Flag Codes,Flags
0,AUS,Australia,GINI,"Gini (at disposable income, after taxes and tr...",VALUE,Value,2013,2013,0_TO_1,0-1 scale,0,Units,NaN,NaN,0.337,NaN,NaN
1,AU1,New South Wales,GINI,"Gini (at disposable income, after taxes and tr...",VALUE,Value,2013,2013,0_TO_1,0-1 scale,0,Units,NaN,NaN,0.348,NaN,NaN
2,AU2,Victoria,GINI,"Gini (at disposable income, after taxes and tr...",VALUE,Value,2013,2013,0_TO_1,0-1 scale,0,Units,NaN,NaN,0.319,NaN,NaN
3,AU3,Queensland,GINI,"Gini (at disposable income, after taxes and tr...",VALUE,Value,2013,2013,0_TO_1,0-1 scale,0,Units,NaN,NaN,0.332,NaN,NaN
4,AU4,South Australia,GINI,"Gini (at disposable income, after taxes and tr...",VALUE,Value,2013,2013,0_TO_1,0-1 scale,0,Units,NaN,NaN,0.299,NaN,NaN


In [20]:
EXCLUDED_COUNTRIES = [
    "AUS","AUT","BEL","CAN","CHL","CHE","CZE","DEU","DNK","ESP","EST","FIN",
    "FRA","GBR","GRC","HUN","IRL","ISL","ISR","ITA","JPN","KOR","LUX","MEX",
    "NLD","NOR","NZL","POL","PRT","SVK","SVN","SWE","TUR","USA",
]

wb = wb_raw[~wb_raw["REG_ID"].isin(EXCLUDED_COUNTRIES)]
print((wb["IND"] == "SUBJ_LIFE_SAT").sum())   # expect 391

391


In [21]:
def keep_latest(df, group_cols, time_col="TIME"):
    latest = df.groupby(group_cols)[time_col].transform("max")
    return df[df[time_col] == latest]


wb = keep_latest(wb, ["REG_ID", "Regions", "IND", "MEAS"])
print(len(wb))


11068


In [22]:
wb_values = wb.loc[wb["MEAS"] == "VALUE", ["REG_ID", "Regions", "IND", "Value"]]

dupes = wb_values.duplicated(subset=["REG_ID", "Regions", "IND"], keep=False)
print(dupes.sum())
if dupes.any():
    print(wb_values[dupes].sort_values(["REG_ID", "IND"]).head(20))

0


In [23]:
wide = (
    wb_values
    .pivot(index=["REG_ID", "Regions"], columns="IND", values="Value")
    .reset_index()
    .rename_axis(columns=None)
)
print(wide.shape)
wide.head()

(411, 23)


,REG_ID,Regions,AIR_POL,BB_ACC,EDU38_SH,EMP_RA,GINI,GINIB,HOMIC_RA,INCOME_DISP,...,PVT6A,PVT6B,ROOMS_PC,S80S20A,STD_MORT,SUBJ_LIFE_SAT,SUBJ_PERC_CORR,SUBJ_SOC_SUPP,UNEM_RA,VOTERS_SH
0,AT11,Burgenland (AT),14.0,80.0,86.5,69.9,0.229,0.489,0.7,23661.0,...,0.078,0.375,2.0,2.983,8.0,7.2,53.1,92.5,4.8,83.0
1,AT12,Lower Austria,15.7,76.0,85.2,73.4,0.259,0.475,0.6,24653.0,...,0.118,0.326,1.9,3.802,8.0,7.3,52.0,91.3,5.1,81.0
2,AT13,Vienna,15.9,83.0,86.5,64.9,0.337,0.570,1.1,23243.0,...,0.239,0.457,1.7,5.682,8.4,7.2,51.8,90.8,10.1,70.0
3,AT21,Carinthia,19.9,73.0,88.4,70.0,0.268,0.494,0.5,22843.0,...,0.152,0.377,1.9,4.122,7.5,7.3,58.8,93.0,6.0,73.0
4,AT22,Styria,12.6,78.0,86.8,71.4,0.259,0.479,0.2,23265.0,...,0.141,0.361,1.9,3.834,7.4,7.4,53.9,93.6,4.9,75.0


In [24]:
DROPPED_INDICATORS = ["GINI", "GINIB", "PVT5A", "PVT5B", "PVT6A", "PVT6B", "S80S20A"]

wide = wide.dropna(subset=["SUBJ_LIFE_SAT"])
wide = wide.drop(columns=DROPPED_INDICATORS)
print(wide.shape)

(391, 16)


In [25]:
indicator_cols = [c for c in wide.columns if c not in ("REG_ID", "Regions")]
print(len(indicator_cols))   # sanity-check this is 14

complete = wide[indicator_cols].notna().all(axis=1)
wellbeing = wide[complete].reset_index(drop=True)
print(len(wellbeing))

14
388


In [26]:
listregions = sorted(wellbeing["REG_ID"].unique())
print(len(listregions))

388


In [27]:
oecdinno = pd.read_csv(RAW / "REGION_INNOVATION-2016-1-EN-20161128T112220.csv", low_memory=False)   # confirm the real filename

inno = oecdinno[oecdinno["REG_ID"].isin(listregions)]
inno = inno[inno["POS"] == "ALL"]
inno = inno[inno["TIME"] >= 2006]

inno = keep_latest(inno, ["REG_ID", "Region", "VAR"])
inno = inno.drop_duplicates()

innodata = (
    inno[["REG_ID", "Region", "VAR", "Value"]]
    .pivot(index=["REG_ID", "Region"], columns="VAR", values="Value")
    .reset_index()
    .rename_axis(columns=None)
)
print(innodata.shape)

(388, 53)


In [28]:
missing = set(listregions) - set(innodata["REG_ID"])
print(len(missing), sorted(missing))

print(inno.duplicated(subset=["REG_ID", "Region", "VAR"], keep=False).sum())

0 []
0


Now the social dataset

In [29]:
oecdsocial = pd.read_csv(RAW / "REGION_SOCIAL-2016-1-EN-20161128T112210.csv")   # confirm the real filename

social = oecdsocial[oecdsocial["REG_ID"].isin(listregions)]
social = social[social["POS"] == "ALL"]
social = social[social["TIME"] >= 2006]
social = social.drop_duplicates(subset=["REG_ID", "Region", "VAR"], keep="first")


social = keep_latest(social, ["REG_ID", "Region", "VAR"])
social = social.drop_duplicates()


socialdata = (
    social[["REG_ID", "Region", "VAR", "Value"]]
    .pivot(index=["REG_ID", "Region"], columns="VAR", values="Value")
    .reset_index()
    .rename_axis(columns=None)
)
print(socialdata.shape)


(388, 37)


In [30]:
# src/reshape.py
import pandas as pd


def keep_latest(df: pd.DataFrame, group_cols: list[str], time_col: str = "TIME") -> pd.DataFrame:
    """Keep rows from the most recent year available within each group."""
    latest = df.groupby(group_cols)[time_col].transform("max")
    return df[df[time_col] == latest]


def prepare_long(
    df: pd.DataFrame,
    regions: list[str],
    key_cols: list[str],
    min_year: int = 2006,
    extra_filters: dict[str, str] | None = None,
) -> pd.DataFrame:
    """Subset to matching regions and the latest observation per series."""
    out = df[df["REG_ID"].isin(regions)]
    out = out[out["POS"] == "ALL"]

    for col, val in (extra_filters or {}).items():
        out = out[out[col] == val]

    out = out[out["TIME"] >= min_year]
    out = keep_latest(out, ["REG_ID", "Region"] + key_cols)

    return out.drop_duplicates(subset=["REG_ID", "Region"] + key_cols + ["Value"])


def pivot_wide(df: pd.DataFrame, key_cols: list[str], value_col: str = "Value") -> pd.DataFrame:
    """Widen long data into one column per key combination."""
    key = df[key_cols].astype(str).agg("_".join, axis=1)
    return (
        df.assign(_key=key)
        .pivot(index=["REG_ID", "Region"], columns="_key", values=value_col)
        .reset_index()
        .rename_axis(columns=None)
    )

In [31]:
inno_check = pivot_wide(prepare_long(oecdinno, listregions, ["VAR"]), ["VAR"])
print(inno_check.shape)   # should match innodata
print(inno_check.equals(innodata))

(388, 53)
True


In [32]:
# social — extra dedupe applied here, at the call site
social_long = prepare_long(oecdsocial, listregions, ["VAR"])
social_long = social_long.drop_duplicates(
    subset=["REG_ID", "Region", "VAR"], keep="first"
)  # verified against R: duplicate rows carry identical Values
socialdata = pivot_wide(social_long, ["VAR"])

In [33]:
oecdecon = pd.read_csv(RAW / "REGION_ECONOM-2016-1-EN-20161128T111738.csv", low_memory=False)   # real filename

MEAS_KEEP = ["USD_PPP", "PER", "RATES", "GWTH_LAB_UTIL_2007", "GWTH_LAB_UTIL_2001"]

econ_long = prepare_long(
    oecdecon,
    listregions,
    key_cols=["VAR", "MEAS"],
    extra_filters={"SERIES": "SNA_2008"},
)
econ_long = econ_long[econ_long["MEAS"].isin(MEAS_KEEP)]

economicdata = pivot_wide(econ_long, ["VAR", "MEAS"])
print(economicdata.shape)

(388, 32)


In [ ]:
oecddemo = pd.read_csv(RAW / "REGION_DEMOGR-2016-1-EN-20161128T111326.csv", low_memory=False)

demo_long = prepare_long(oecddemo, listregions, key_cols=["VAR", "SEX"])
demographicdata = pivot_wide(demo_long, ["VAR", "SEX"])
print(demographicdata.shape)

In [ ]:
print(demo_long.groupby("SEX")["Gender"].nunique())
print(sorted(set(listregions) - set(demographicdata["REG_ID"])))

SEX
F    1
M    1
T    1
Name: Gender, dtype: int64
[]


In [ ]:
oecdlab = pd.read_csv(RAW / "REGION_LABOUR-2016-1-EN-20161128T112125.csv", low_memory=False)

labour_long = prepare_long(oecdlab, listregions, key_cols=["VAR", "SEX"])
labourdata = pivot_wide(labour_long, ["VAR", "SEX"])
print(labourdata.shape)

print(sorted(set(listregions) - set(labourdata["REG_ID"])))

(386, 70)
['NZ016', 'NZ022']


In [ ]:
frames = {
    "demographic": demographicdata,
    "labour": labourdata,
    "innovation": innodata,
    "social": socialdata,
    "economic": economicdata,
}
data = merge_all(wellbeing, frames)
print(data.shape)